# Automation & Agentic AI — Practical Notebook (LangChain edition)
## STUDENT VERSION — fill in every `TODO`

**A hands-on companion to the session on automation and agentic AI, built with LangChain.**

---
### What you'll do in this notebook

- Connect to Google Gemini through LangChain.
- Look at a **worked example**: a 3-agent chain that turns a topic into a
  short report (Research → Summarize → Format).
- Complete the session's exercise: **Design a simple AI-assisted workflow**
  — then **build it** as your own chain of 2-3 agents.

### A few words explained up front

- **Chain** — a sequence of steps piped together with `|`. The output of one
  step becomes the input of the next.
- **Agent (in this notebook)** — a chain with a specific role (e.g.
  "Summarizer"), built from a prompt + an LLM (+ optionally a tool). One
  agent in the worked example also uses a real tool (web search) — that's
  the difference between a "chain" and a proper tool-using "agent" in the
  strict sense, and you'll see both.
- **LCEL (`|`)** — the pipe operator that connects steps: `prompt | llm | parser`
  reads left to right, like a small production line.

### How to use this notebook

Cells marked **`# TODO`** are yours to complete. Everything else is given —
read it, run it, and it should just work. Run cells **in order, top to
bottom** — later cells reuse the `llm` connection and helper functions set
up earlier.

> **How to run this:** Google Colab or a normal Jupyter notebook both work.
> No GPU needed. You'll need a free Gemini API key (instructions below).

## 1 · Install the libraries

Run this cell first.

In [1]:
!pip install -q langchain langchain-google-genai langgraph ddgs

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Windows\\system32\\odysseus\\venv\\Lib\\site-packages\\filetype'
Check the permissions.


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Windows\system32\odysseus\venv\Scripts\python.exe -m pip install --upgrade pip


## 2 · Connect to the Gemini AI model

Get a free API key at **https://aistudio.google.com/apikey**

In [2]:
import os
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL = "gemini-3.5-flash-lite"

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key here: ")

# This "llm" object is our connection to the AI model.
# Every agent/chain we build below will reuse this same connection.
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.4)

print("Connected! Using model:", MODEL)

Paste your Gemini API key here:  ········


Connected! Using model: gemini-3.5-flash-lite


## 3 · What does "chaining agents" mean in LangChain?

A single agent in this notebook is just three pieces glued together:

```
   a PROMPT (its instructions)  -->  the LLM  -->  an OUTPUT PARSER (cleans up the result)
```

Written in LangChain, that's literally: `prompt | llm | parser`. Read the `|`
as "then send the output of this into that" — the same idea as a Unix pipe.

**Chaining multiple agents** just means connecting several of these in a row,
where each one has its own role:

```
  Agent 1 (Research)  -->  Agent 2 (Analyze)  -->  Agent 3 (Write)
   raw information          prioritized facts +        flowing narrative
                             why each one matters       report, no bullets
```

This is a simple version of the "Orchestrator → Workers" pattern from the
session slides — except here the hand-off is a straight line instead of a
central orchestrator. Section 4 builds exactly this, with 3 agents.

## 4 · Worked example — a 3-agent chained workflow (given)

**The task:** turn a topic into a short, polished report.

- **Agent 1 — Researcher.** A real tool-using agent (it can call a web
  search tool) that gathers a few raw facts.
- **Agent 2 — Analyst.** A plain prompt-chain that doesn't just shorten the
  raw facts — it picks the most important ones and explains *why each one
  matters*, so there's real reasoning happening here, not just compression.
- **Agent 3 — Writer.** A plain prompt-chain that turns that analysis into a
  short **narrative report** — flowing sentences, not a list. This is a
  genuinely different kind of output than Agent 2 produces, which is the
  whole point of having a separate stage: each agent should transform its
  input into something meaningfully different, not just reformat it.

### 4.1 · Agent 1 — a tool-using Research agent

This one is a real agent in the strict sense: it can decide, on its own, to
call the `web_search` tool before answering. We build it with LangChain's
`create_agent` — the standard way to get a tool-calling agent without
writing the reasoning loop yourself (same idea as CrewAI's `Agent`, different
library).

In [6]:
!pip install ddgs

  Using cached ddgs-9.15.0-py3-none-any.whl.metadata (16 kB)
  Using cached fake_useragent-2.2.0-py3-none-any.whl.metadata (17 kB)
  Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
  Using cached h2-4.4.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached socksio-1.0.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached hpack-4.2.0-py3-none-any.whl.metadata (3.3 kB)
Using cached ddgs-9.15.0-py3-none-any.whl (50 kB)
Using cached fake_useragent-2.2.0-py3-none-any.whl (161 kB)
Using cached h2-4.4.1-py3-none-any.whl (62 kB)
Using cached hpack-4.2.0-py3-none-any.whl (34 kB)
Using cached hyperframe-6.1.0-py3-none-any.whl (13 kB)
Using cached socksio-1.0.0-py3-none-any.whl (12 kB)
Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl (369 kB)

   ---------------------------------------- 0/7 [brotli]



ERROR: Could not install packages due to an OSError: [Errno 13] Permission denied: 'C:\\Windows\\system32\\odysseus\\venv\\Lib\\site-packages\\_brotli.cp313-win_amd64.pyd'
Check the permissions.


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Windows\system32\odysseus\venv\Scripts\python.exe -m pip install --upgrade pip


In [8]:
from langchain_core.tools import tool
from ddgs import DDGS

@tool
def web_search(query: str) -> str:
    """Search the web and return a few short results with titles and snippets."""
    try:
        results = DDGS().text(query, max_results=3)
        if not results:
            return f"No results found for '{query}'."
        return "\n".join(f"- {r['title']}: {r['body'][:180]}" for r in results)
    except Exception as e:
        return f"Search failed for '{query}' ({e}). Try again in a moment."

# Sanity check -- call the tool directly, no agent involved yet.
print(web_search.invoke("population of Japan"))

ModuleNotFoundError: No module named 'ddgs'

In [ ]:
from langchain.agents import create_agent

research_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=(
        "You are a research agent. Use the web_search tool to gather a "
        "few raw facts about the topic you're given. Keep it brief and "
        "factual -- no formatting, no opinions, just the facts you found."
    ),
)

def run_research_agent(topic: str) -> str:
    """Run the research agent on a topic and return its final text answer."""
    result = research_agent.invoke({
        "messages": [{"role": "user", "content": f"Research this topic: {topic}"}]
    })
    return result["messages"][-1].content

# Sanity check
print(run_research_agent("the current price of Bitcoin in USD"))

### 4.2 · Agent 2 — an Analyst (plain chain, no tools)

Not every agent needs a tool. This one just needs a clear role and a focused
prompt. Notice the shape: `prompt | llm | parser` -- that's the whole agent.

Its job is deliberately **more than shortening text**: it has to *decide*
which facts matter most and *explain why* -- that's a small piece of
reasoning, not just compression. That's what makes it worth being its own
agent instead of a copy-paste step.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

summarizer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an analysis agent. You'll be given raw, unstructured research "
     "notes. Do three things:\n"
     "1. Pick the 3 most important facts -- drop anything trivial or repeated.\n"
     "2. For each fact, add a short 'why it matters' explanation in the same line.\n"
     "3. If anything in the notes looks uncertain, outdated, or contradictory, "
     "flag it in one line at the end starting with 'Note:'.\n"
     "Output as a numbered list: `1. <fact> -- <why it matters>`. This is "
     "intermediate output for another agent, not the final report, so skip "
     "any greetings or closing remarks."),
    ("human", "{raw_notes}"),
])

summarizer_chain = summarizer_prompt | llm | StrOutputParser()

# Sanity check -- test with made-up raw notes, no research agent involved.
# Notice the output isn't just shorter -- it explains why each fact matters.
print(summarizer_chain.invoke({"raw_notes": "Cats sleep 12-16 hours a day. "
                                             "Cats have 32 muscles in each ear. "
                                             "A group of cats is called a clowder."}))

### 4.3 · Agent 3 — a Writer (plain chain, no tools)

Same shape again -- a different role and prompt -- but this time the job is
to genuinely **rewrite**, not just restyle. Agent 2 hands over a numbered,
analytical list; Agent 3's job is to turn that into flowing prose a person
would actually enjoy reading. If this agent's output looked just like Agent
2's with a sentence tacked on, it wouldn't be earning its place in the
chain -- so its prompt explicitly forbids lists and bullets.

In [ ]:
formatter_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a writing agent. You'll be given a short numbered analysis "
     "(facts plus why each one matters). Rewrite it as a short narrative "
     "report for a general reader: 3-4 flowing, connected sentences. "
     "Open with a one-line hook, weave the facts and why they matter "
     "naturally into the prose, and end with one forward-looking or "
     "takeaway sentence. Do NOT use a list, bullet points, numbers, or "
     "headers -- plain paragraph prose only."),
    ("human", "{summary}"),
])

formatter_chain = formatter_prompt | llm | StrOutputParser()

# Sanity check -- feed it Agent 2's *style* of output (a numbered analysis)
# and confirm you get back prose, not the same list restated.
print(formatter_chain.invoke({"summary":
    "1. Cats sleep 12-16 hours a day -- indoor cats need engaging play during "
    "their shorter waking hours.\n"
    "2. Cats have 32 muscles in each ear -- this gives them excellent "
    "directional hearing for hunting.\n"
    "3. A group of cats is called a 'clowder' -- a fun fact people rarely know."
}))

### 4.4 · Run the whole workflow, step by step

Now chain all three agents by hand: run Agent 1, feed its output into
Agent 2, feed *that* output into Agent 3.

In [ ]:
topic = "agentic AI being used in classrooms"

raw_notes = run_research_agent(topic)
print("=== AGENT 1 (Research) ===")
print(raw_notes)

summary = summarizer_chain.invoke({"raw_notes": raw_notes})
print("\n=== AGENT 2 (Summarize) ===")
print(summary)

final_report = formatter_chain.invoke({"summary": summary})
print("\n=== AGENT 3 (Format) -- FINAL ANSWER ===")
print(final_report)

### Bonus (given) — the same workflow as one LCEL chain

Everything above can be written as a **single piped expression** — this is
the "chain" in LangChain. `RunnableLambda` just wraps a plain function so it
can sit inside a `|` pipeline alongside the prompt-chains.

In [ ]:
from langchain_core.runnables import RunnableLambda

full_workflow = (
    RunnableLambda(lambda topic: run_research_agent(topic))
    | RunnableLambda(lambda raw_notes: {"raw_notes": raw_notes})
    | summarizer_chain
    | RunnableLambda(lambda summary: {"summary": summary})
    | formatter_chain
)

result = full_workflow.invoke("agentic AI being used in classrooms")
print(result)

Same result, one expression. Both versions are "correct" LangChain — the
step-by-step version is easier to debug (you can print each stage), the
piped version is more idiomatic once you trust each piece. Use whichever
you prefer for your own workflow below.

## 5 · Exercise — Design a simple AI-assisted workflow

Pick **one small task** you actually do, that naturally breaks into 2-3
stages where each stage has a clearly different job. That's what makes it a
good fit for chaining. Some ideas:

- Raw meeting notes → extracted action items → a clean checklist message
- A list of messy data → cleaned-up data → a short written summary
- A question → researched facts → a beginner-friendly explanation

Fill in the blanks below. This is planning only — no agents built yet.

In [ ]:
# TODO 5.1 -- Fill in every blank with your own answers.

task_name = "..."                  # e.g. "Meeting notes to action items"

# List your agents IN ORDER. Each one should have a narrow, clear job --
# that's what makes it easy to write a good prompt for it.
agent_plan = [
    {"agent_name": "...", "job": "..."},   # e.g. {"agent_name": "Extractor", "job": "pull out action items from raw notes"}
    {"agent_name": "...", "job": "..."},   # e.g. {"agent_name": "Formatter", "job": "turn action items into a checklist"}
]

does_any_agent_need_a_tool = "..."   # "yes" or "no" -- and if yes, which one?

one_guardrail = "..."                # TODO: one safety limit you'd add,
                                      # e.g. "always show the draft before sending"

print("Task:", task_name)
for a in agent_plan:
    print(" -", a["agent_name"], "->", a["job"])
print("Needs a tool?", does_any_agent_need_a_tool)
print("Guardrail:", one_guardrail)

## 6 · Build your chained-agent workflow

Build **at least 2 agents** from your `agent_plan` above, following the exact
pattern from Section 4: a `ChatPromptTemplate`, piped into `llm`, piped into
`StrOutputParser()`. A tool is optional — copy the Section 4.1 pattern only
if your design actually needs one.

#### TODO 6.1 — Build Agent 1 from your plan

In [ ]:
# TODO 6.1 -- Build your first agent. Copy the shape of `summarizer_chain`
# from Section 4.2 (or `research_agent` from 4.1, if this stage needs a tool).

agent1_prompt = ChatPromptTemplate.from_messages([
    ("system", "..."),   # TODO: describe this agent's one job, clearly
    ("human", "{...}"),  # TODO: name the input variable this agent expects
])

agent1_chain = agent1_prompt | llm | StrOutputParser()

# Sanity check -- test it on its own with made-up input before chaining anything.
print(agent1_chain.invoke({"...": "..."}))   # TODO: match the key you used above

#### TODO 6.2 — Build Agent 2 from your plan

This one should take Agent 1's *output* as its input.

In [ ]:
# TODO 6.2 -- Build your second agent, same pattern as 6.1.

agent2_prompt = ChatPromptTemplate.from_messages([
    ("system", "..."),   # TODO
    ("human", "{...}"),  # TODO
])

agent2_chain = agent2_prompt | llm | StrOutputParser()

# Sanity check
print(agent2_chain.invoke({"...": "..."}))   # TODO

#### (Optional) TODO 6.3 — Build a third agent

Only if your `agent_plan` has 3 stages. Skip this cell if 2 agents cover
your task.

In [ ]:
# TODO 6.3 (optional) -- Build a third agent, same pattern, if your plan needs one.

agent3_prompt = ChatPromptTemplate.from_messages([
    ("system", "..."),   # TODO
    ("human", "{...}"),  # TODO
])

agent3_chain = agent3_prompt | llm | StrOutputParser()

print(agent3_chain.invoke({"...": "..."}))   # TODO

#### TODO 6.4 — Run your full chain, step by step

Copy the pattern from Section 4.4: run agent 1, feed its output into agent 2
(and agent 3, if you built one), print each stage.

In [ ]:
# TODO 6.4 -- Run your agents in order, passing each output to the next input.

my_input = "..."   # TODO: a real example input for your task

step1_output = agent1_chain.invoke({"...": my_input})   # TODO: match your key
print("=== AGENT 1 ===")
print(step1_output)

step2_output = agent2_chain.invoke({"...": step1_output})   # TODO: match your key
print("\n=== AGENT 2 -- FINAL ANSWER (or pass to agent 3) ===")
print(step2_output)

# TODO (optional): if you built agent3_chain, add the same pattern here.

## 7 · Wrap-up (written answer)

Edit this cell:

1. Did splitting the task across multiple agents actually help, or would
   one agent with a longer prompt have done just as well?
2. Would any stage benefit from a real tool (like the Research agent's web
   search)? Which one, and why?
3. Looking back at your `one_guardrail` from Section 5 — where exactly would
   you add it in the chain you just built?

*(Your answers here)*